# 8.3 知识蒸馏 (Knowledge Distillation)

知识蒸馏将大模型（教师）的知识迁移到小模型（学生），使小模型在保持效率的同时获得接近大模型的性能。

本节涵盖：
- 经典蒸馏（Hinton 2015）
- 中间层蒸馏（TinyBERT/MiniLM）
- LLM蒸馏（黑盒/白盒/合成数据）
- 渐进式蒸馏
- 任务特定蒸馏
- 蒸馏最佳实践

## 1. 经典蒸馏 (Hinton 2015)

### 核心思想
教师模型输出的概率分布（软标签）包含比硬标签更丰富的信息——类间关系（"暗知识"）。

### 核心公式
- **总损失**：$L = \alpha \cdot L_{hard} + (1-\alpha) \cdot L_{soft}$
- **软标签损失**：$L_{soft} = \text{KL}(\sigma(z_s/T) \| \sigma(z_t/T)) \cdot T^2$
- **温度T**：控制分布平滑度，T越大暴露越多类间关系

### 温度效应
| T值 | 效果 | 适用场景 |
|------|------|----------|
| T=1 | 标准softmax | 硬标签训练 |
| T=2-5 | 适度平滑 | 通用蒸馏 |
| T=10+ | 极度平滑 | 暗知识最大化 |
| T→∞ | 均匀分布 | 无信息 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.7, temperature=4.0):
        super().__init__()
        self.alpha = alpha
        self.temperature = temperature

    def forward(self, student_logits, teacher_logits, labels):
        soft_loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction='batchmean'
        ) * (self.temperature ** 2)
        hard_loss = F.cross_entropy(student_logits, labels)
        total = self.alpha * hard_loss + (1 - self.alpha) * soft_loss
        return total, hard_loss, soft_loss

class TeacherModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x):
        return self.net(x)

class StudentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(self, x):
        return self.net(x)

teacher = TeacherModel()
student = StudentModel()
distill_loss = DistillationLoss(alpha=0.7, temperature=4.0)

x = torch.randn(16, 64)
labels = torch.randint(0, 10, (16,))

with torch.no_grad():
    teacher_logits = teacher(x)
student_logits = student(x)

total, hard, soft = distill_loss(student_logits, teacher_logits, labels)

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print('=== Classic Knowledge Distillation ===')
print(f'Teacher: {teacher_params:,} params, Student: {student_params:,} params ({student_params/teacher_params:.1%})')
print(f'\nLosses: total={total.item():.4f}, hard={hard.item():.4f}, soft={soft.item():.4f}')

print(f'\nTemperature effect on distribution (sample 0):')
for T in [1, 2, 4, 10]:
    probs = F.softmax(teacher_logits[0] / T, dim=-1)
    entropy = -(probs * probs.log()).sum().item()
    top3 = probs.topk(3)
    print(f'  T={T:2d}: entropy={entropy:.3f}, top3_probs={[f"{p:.3f}" for p in top3.values.tolist()]}')

print(f'\nKey: Higher T -> smoother distribution -> more "dark knowledge" exposed.')
print(f'T² scaling prevents soft loss from being too small when T is large.')

## 2. 中间层蒸馏

不仅蒸馏最终输出，还让学生模型的中间层表征匹配教师模型的中间层。

### TinyBERT方法
1. **注意力层蒸馏**：学生注意力矩阵匹配教师注意力矩阵
2. **隐藏层蒸馏**：学生隐藏状态线性变换后匹配教师隐藏状态
3. **嵌入层蒸馏**：学生嵌入匹配教师嵌入
4. **预测层蒸馏**：经典logit蒸馏

### MiniLM方法
只蒸馏最后一层的注意力关系矩阵(Q·K^T和attn·V)，不需要层对齐。

### 对比
| 方法 | 蒸馏目标 | 需要层对齐 | 效果 |
|------|---------|-----------|------|
| 经典 | Logits | 否 | 基础 |
| TinyBERT | Attn+Hidden+Logits | 是 | 强 |
| MiniLM | Attn关系矩阵 | 否 | 强 |
| PKT | 概率分布 | 否 | 中 |

In [ ]:
class IntermediateDistillation(nn.Module):
    def __init__(self, d_teacher=128, d_student=32, n_classes=10, temperature=4.0, alpha=0.5):
        super().__init__()
        self.d_teacher = d_teacher
        self.d_student = d_student
        self.temperature = temperature
        self.alpha = alpha
        self.teacher = nn.Sequential(
            nn.Linear(64, d_teacher), nn.ReLU(),
            nn.Linear(d_teacher, d_teacher), nn.ReLU(),
            nn.Linear(d_teacher, n_classes)
        )
        self.student = nn.Sequential(
            nn.Linear(64, d_student), nn.ReLU(),
            nn.Linear(d_student, d_student), nn.ReLU(),
            nn.Linear(d_student, n_classes)
        )
        self.proj1 = nn.Linear(d_student, d_teacher)
        self.proj2 = nn.Linear(d_student, d_teacher)

    def forward_teacher(self, x):
        h1 = self.teacher[0](x)
        h1_act = self.teacher[1](h1)
        h2 = self.teacher[2](h1_act)
        h2_act = self.teacher[3](h2)
        out = self.teacher[4](h2_act)
        return out, [h1_act, h2_act]

    def forward_student(self, x):
        h1 = self.student[0](x)
        h1_act = self.student[1](h1)
        h2 = self.student[2](h1_act)
        h2_act = self.student[3](h2)
        out = self.student[4](h2_act)
        h1_proj = self.proj1(h1_act)
        h2_proj = self.proj2(h2_act)
        return out, [h1_proj, h2_proj]

    def compute_loss(self, x, labels):
        t_out, t_hidden = self.forward_teacher(x)
        s_out, s_hidden = self.forward_student(x)
        logit_loss = F.kl_div(
            F.log_softmax(s_out / self.temperature, dim=-1),
            F.softmax(t_out.detach() / self.temperature, dim=-1),
            reduction='batchmean'
        ) * (self.temperature ** 2)
        hidden_losses = [F.mse_loss(sh, th.detach()) for sh, th in zip(s_hidden, t_hidden)]
        total_hidden = sum(hidden_losses) / len(hidden_losses)
        total = self.alpha * logit_loss + (1 - self.alpha) * total_hidden
        return total, logit_loss, total_hidden

distill = IntermediateDistillation(d_teacher=128, d_student=32, n_classes=10)
optimizer = torch.optim.Adam(distill.parameters(), lr=1e-3)
x = torch.randn(16, 64)
labels = torch.randint(0, 10, (16,))

print('=== Intermediate Layer Distillation ===')
t_params = sum(p.numel() for p in distill.teacher.parameters())
s_params = sum(p.numel() for p in distill.student.parameters())
print(f'Teacher: {t_params:,} params, Student: {s_params:,} params ({s_params/t_params:.1%})')

for epoch in range(10):
    total, logit_l, hidden_l = distill.compute_loss(x, labels)
    optimizer.zero_grad()
    total.backward()
    optimizer.step()
    if (epoch + 1) % 3 == 0:
        with torch.no_grad():
            s_out, _ = distill.forward_student(x)
            acc = (s_out.argmax(1) == labels).float().mean()
        print(f'Epoch {epoch+1}: total={total.item():.4f}, logit={logit_l.item():.4f}, hidden={hidden_l.item():.4f}, acc={acc:.2%}')

print(f'\nKey: Intermediate layer distillation provides richer supervision than logit-only.')
print(f'Projection layers align student and teacher hidden dimensions.')

## 3. LLM蒸馏

### 黑盒蒸馏
只用教师模型的输出（文本/概率）作为训练数据，不需要访问模型内部。
- **Alpaca**：GPT-4生成指令跟随数据，训练LLaMA
- **Vicuna**：用户对话数据，GPT-4评分
- **Orca**：渐进式复杂度，从简单到困难

### 白盒蒸馏
访问教师模型的logits和中间层，提供更丰富的监督信号。
- **MiniLLM**：KL散度反转，用教师分布采样训练学生
- **GKD** (Generalized Knowledge Distillation)：on-policy采样

### 合成数据蒸馏
用大模型生成高质量训练数据，训练小模型。
- **Self-Instruct**：模型自我生成指令数据
- **Evol-Instruct**：逐步进化指令复杂度
- **Magpie**：基于对话模板自动生成数据

### 产业实践
| 项目 | 教师 | 学生 | 方法 |
|------|------|------|------|
| Alpaca | GPT-4 | LLaMA-7B | 黑盒SFT |
| Vicuna | GPT-4 | LLaMA-13B | 黑盒SFT |
| Orca | GPT-4 | LLaMA-13B | 渐进式黑盒 |
| MiniLLM | LLaMA-65B | LLaMA-7B | 白盒KL |
| Phi | GPT-4 | 1.3B/2.7B | 合成数据 |

In [ ]:
class BlackBoxDistillation:
    def __init__(self, teacher, student, temperature=2.0):
        self.teacher = teacher
        self.student = student
        self.temperature = temperature
        self.synthetic_data = []

    def generate_synthetic_data(self, n_samples=100, prompt_dim=64):
        for _ in range(n_samples):
            prompt = torch.randn(1, prompt_dim)
            with torch.no_grad():
                teacher_output = self.teacher(prompt)
            self.synthetic_data.append((prompt, teacher_output))
        return len(self.synthetic_data)

    def train_step(self, prompts, teacher_outputs, labels=None):
        student_outputs = self.student(prompts)
        soft_loss = F.kl_div(
            F.log_softmax(student_outputs / self.temperature, dim=-1),
            F.softmax(teacher_outputs / self.temperature, dim=-1),
            reduction='batchmean'
        ) * (self.temperature ** 2)
        if labels is not None:
            hard_loss = F.cross_entropy(student_outputs, labels)
            return 0.5 * hard_loss + 0.5 * soft_loss
        return soft_loss

class ProgressiveDistillation:
    def __init__(self, teacher, student_sizes=[32, 64, 128], temperature=2.0):
        self.teacher = teacher
        self.student_sizes = student_sizes
        self.temperature = temperature
        self.students = []
        for size in student_sizes:
            self.students.append(nn.Sequential(
                nn.Linear(64, size), nn.ReLU(), nn.Linear(size, 10)
            ))

    def train_progressive(self, x, labels, n_steps=5):
        results = []
        for i, student in enumerate(self.students):
            optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)
            for step in range(n_steps):
                s_out = student(x)
                with torch.no_grad():
                    t_out = self.teacher(x)
                loss = F.kl_div(
                    F.log_softmax(s_out / self.temperature, dim=-1),
                    F.softmax(t_out / self.temperature, dim=-1),
                    reduction='batchmean'
                ) * (self.temperature ** 2)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            with torch.no_grad():
                acc = (student(x).argmax(1) == labels).float().mean()
            results.append({'size': self.student_sizes[i], 'acc': acc.item()})
        return results

teacher = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
student = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

bb_distill = BlackBoxDistillation(teacher, student)
n_data = bb_distill.generate_synthetic_data(n_samples=50)

print('=== Black-Box Distillation ===')
print(f'Generated {n_data} synthetic data samples from teacher')

x = torch.randn(16, 64)
labels = torch.randint(0, 10, (16,))
with torch.no_grad():
    t_out = teacher(x)
loss = bb_distill.train_step(x, t_out, labels)
print(f'Distillation loss: {loss.item():.4f}')

prog = ProgressiveDistillation(teacher, student_sizes=[16, 32, 64])
results = prog.train_progressive(x, labels, n_steps=10)

print(f'\n=== Progressive Distillation ===')
for r in results:
    params = sum(p.numel() for p in prog.students[results.index(r)].parameters())
    print(f'  Hidden={r["size"]}: {params:,} params, acc={r["acc"]:.2%}')

print(f'\nKey: Black-box distillation only needs teacher outputs (no internal access).')
print(f'Progressive distillation trains students of increasing size for size-accuracy trade-off.')

## 4. 注意力蒸馏与关系蒸馏

### 注意力蒸馏
让学生模型的注意力分布匹配教师模型的注意力分布。
损失函数：$L_{attn} = \frac{1}{h}\sum_{i=1}^{h} \text{KL}(A_s^i \| A_t^i)$

### 关系知识蒸馏 (RKD)
不蒸馏单个样本的输出，而是蒸馏样本间的关系（距离、角度）。
- **距离损失**：$L_{dist} = \sum_{i<j} |d_t(z_i, z_j) - d_s(z_i, z_j)|$
- **角度损失**：$L_{angle} = \sum_{i<j<k} |\theta_t(z_i, z_j, z_k) - \theta_s(z_i, z_j, z_k)|$

### 优势
- 不要求学生和教师有相同的架构
- 关注样本间关系而非绝对值，更鲁棒
- 适合跨架构蒸馏

In [ ]:
class AttentionDistillation(nn.Module):
    def __init__(self, d_model=64, n_heads=4, d_head=16):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_head
        self.teacher_qkv = nn.Linear(d_model, 3 * n_heads * d_head)
        self.student_qkv = nn.Linear(d_model, 3 * 2 * d_head)

    def get_attention(self, qkv_proj, x, n_heads):
        B, T, D = x.shape
        qkv = qkv_proj(x).reshape(B, T, 3, n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k = qkv[0], qkv[1]
        attn = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        return F.softmax(attn, dim=-1)

    def forward(self, x):
        t_attn = self.get_attention(self.teacher_qkv, x, self.n_heads)
        s_attn = self.get_attention(self.student_qkv, x, 2)
        return t_attn, s_attn

    def attention_loss(self, t_attn, s_attn):
        n = min(t_attn.size(1), s_attn.size(1))
        t_attn_sel = t_attn[:, :n]
        s_attn_sel = s_attn[:, :n]
        loss = F.kl_div(
            F.log_softmax(s_attn_sel.flatten(1), dim=-1),
            F.softmax(t_attn_sel.flatten(1).detach(), dim=-1),
            reduction='batchmean'
        )
        return loss

class RelationalDistillation(nn.Module):
    def __init__(self, d_teacher=128, d_student=32, n_classes=10):
        super().__init__()
        self.teacher = nn.Sequential(nn.Linear(64, d_teacher), nn.ReLU(), nn.Linear(d_teacher, n_classes))
        self.student = nn.Sequential(nn.Linear(64, d_student), nn.ReLU(), nn.Linear(d_student, n_classes))
        self.proj_t = nn.Linear(d_teacher, d_student)

    def distance_potential(self, features):
        n = features.size(0)
        dists = torch.cdist(features, features, p=2)
        return dists / dists.max()

    def angle_potential(self, features):
        n = features.size(0)
        angles = []
        for i in range(min(n, 8)):
            for j in range(i+1, min(n, 8)):
                for k in range(j+1, min(n, 8)):
                    v1 = features[j] - features[i]
                    v2 = features[k] - features[i]
                    cos = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0))
                    angles.append(cos)
        if angles:
            return torch.stack(angles)
        return torch.tensor([0.0])

    def rkd_loss(self, x):
        with torch.no_grad():
            t_feat = self.teacher[0](x)
            t_feat_proj = self.proj_t(t_feat)
        s_feat = self.student[0](x)
        t_dist = self.distance_potential(t_feat_proj)
        s_dist = self.distance_potential(s_feat)
        dist_loss = F.mse_loss(s_dist, t_dist)
        t_angle = self.angle_potential(t_feat_proj)
        s_angle = self.angle_potential(s_feat)
        angle_loss = F.mse_loss(s_angle, t_angle)
        return dist_loss, angle_loss

attn_distill = AttentionDistillation(d_model=64, n_heads=4, d_head=16)
x = torch.randn(4, 8, 64)
t_attn, s_attn = attn_distill(x)
attn_loss = attn_distill.attention_loss(t_attn, s_attn)

print('=== Attention Distillation ===')
print(f'Teacher attention: {t_attn.shape}')
print(f'Student attention: {s_attn.shape}')
print(f'Attention distillation loss: {attn_loss.item():.4f}')

rkd = RelationalDistillation(d_teacher=128, d_student=32)
x = torch.randn(8, 64)
dist_loss, angle_loss = rkd.rkd_loss(x)

print(f'\n=== Relational Knowledge Distillation ===')
print(f'Distance loss: {dist_loss.item():.4f}')
print(f'Angle loss: {angle_loss.item():.4f}')
print(f'\nKey: Attention distillation transfers attention patterns.')
print(f'RKD distills sample relationships, not individual outputs.')
print(f'Both work across different architectures.')

## 5. 蒸馏最佳实践与总结

### 方法选择指南

| 场景 | 推荐方法 | 原因 |
|------|---------|------|
| 同架构压缩 | 经典+中间层 | 层对齐容易 |
| 跨架构压缩 | RKD+注意力 | 不需要层对齐 |
| LLM黑盒 | 合成数据SFT | 无法访问内部 |
| LLM白盒 | MiniLM/GKD | 更高精度 |
| 极致压缩 | 渐进式蒸馏 | 逐步压缩更稳定 |

### 训练技巧
1. **数据增强**：蒸馏数据量比数据质量更重要
2. **温度调度**：从高T开始，逐渐降低
3. **损失权重**：α从0（纯软标签）开始，逐渐增加硬标签权重
4. **教师选择**：更大的教师不一定更好，适中最佳
5. **数据过滤**：过滤教师低置信度的样本

### 常见陷阱
- **容量差距过大**：学生太小无法学习教师知识
- **过拟合软标签**：学生记住教师的错误
- **分布不匹配**：蒸馏数据与实际使用场景不同
- **忽略硬标签**：纯软标签训练在某些任务上不如混合

### 蒸馏总结

| 方法 | 监督信号 | 需要白盒 | 精度提升 |
|------|---------|---------|----------|
| 经典KD | Logits | 是 | 基础 |
| TinyBERT | Attn+Hidden+Logits | 是 | 强 |
| MiniLM | Attn关系 | 是 | 强 |
| RKD | 样本关系 | 是 | 中 |
| 黑盒SFT | 文本输出 | 否 | 中 |
| 合成数据 | 生成数据 | 否 | 中-强 |
| 渐进式 | 逐步增加 | 是 | 最强 |